# Stoa 04 — data bias detection by slicing

Checks whether some groups of rows (**slices**) are under-represented or behave differently in the data. This is the data side of bias detection; checking a model's error per slice needs a trained model and comes later. Reads the two final datasets from **`03_preprocess.ipynb`**.

There are no people-level columns (age, gender, ...) in this data, on purpose: the project uses no personal data (`CLAUDE.md` core rule 6). The groups that matter here are **places, times and conditions**: a model that is accurate on busy CBD afternoons can still be poor on quiet streets, at night, in the rain or in a precinct it never saw.

Slices are for **checking**, not features: a slice column like the precinct or sensor volume is never fed to the model (core rule 9 bans sensor IDs, coordinates and calendar month as features).

Per slice it measures: share of rows, number of sensors, the count distribution, zeros, outage flags and missing weather (sections 3-4), then coverage over time and per sensor (section 5) and the flagged slices (section 6).

Outputs go to `reports/bias/` in the data folder.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

# Same data folder as notebooks 01-03. Colab: your Google Drive. A local clone: data/notebooks/.
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/stoa/data")
else:
    DATA = Path(os.environ.get("STOA_NOTEBOOK_DATA", Path.cwd().resolve().parent / "data" / "notebooks"))
print("data folder:", DATA)

In [ ]:
import json

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

FINAL = DATA / "final"
INTERIM = DATA / "interim"
REPORTS = DATA / "reports" / "bias"
REPORTS.mkdir(parents=True, exist_ok=True)
TZ = "Australia/Melbourne"

# Within-Melbourne transfer precincts, from docs/PROJECT_CONTEXT.md section 7.1.
# Every other sensor is "CBD and other".
PRECINCTS = {
    "UniMelb edge": [42, 43, 44],
    "Carlton": [31, 37, 46, 50, 54],
    "North Melbourne": [70, 76, 85, 86, 87, 180],
}

# Era breaks (Melbourne local dates), the same ones the team's counts_hourly table uses.
COVID_START, COVID_END = "2020-03-16", "2021-10-21"
METRO_TUNNEL = "2025-11-30"

RAIN_MM = 0.2   # an hour counts as "wet" above this much rain

# A slice is flagged when it holds less than SMALL_SHARE of the rows, when its median count is
# more than MEDIAN_RATIO times above or below the overall median, or when more than
# MISSING_WEATHER of its rows have no weather.
SMALL_SHARE = 0.05
MEDIAN_RATIO = 2.0
MISSING_WEATHER = 0.05

### Check the inputs are there

This notebook needs the outputs of `03_preprocess.ipynb` (the current version, which writes `sensor_steepness.parquet` and the `suspect_outage_zero` column) and combine's `missing_hours.parquet`.

In [ ]:
NEEDED = [FINAL / "pcs_hourly.parquet", FINAL / "sensor_steepness.parquet", INTERIM / "missing_hours.parquet"]
missing = [p for p in NEEDED if not p.exists()]
if missing:
    raise FileNotFoundError("missing: " + ", ".join(str(p) for p in missing)
                            + " -- run 02_combine and 03_preprocess first (same data folder)")
cols_needed = {"location_id", "date", "time", "pedestriancount", "source", "suspect_outage_zero", "temp_c", "rain_mm"}
have = set(pq.read_schema(FINAL / "pcs_hourly.parquet").names)
if cols_needed - have:
    raise KeyError(f"pcs_hourly.parquet is missing {sorted(cols_needed - have)}; re-run 03_preprocess (current version)")
print("inputs OK:", ", ".join(p.name for p in NEEDED))

## 2. Load the final datasets

`time` is read as whole UTC hours straight from the parquet file (no slow per-row conversion), then turned back into one UTC timestamp, from which Melbourne local time is worked out. Local time is used for the slices because people follow the local clock (a "morning peak" is 8 a.m. Melbourne time, all year).

In [ ]:
cols = ["location_id", "date", "time", "pedestriancount", "source", "suspect_outage_zero", "temp_c", "rain_mm"]
t = pq.read_table(FINAL / "pcs_hourly.parquet", columns=cols)
hour_utc = pc.divide(t["time"].cast(pa.int64()), 3_600_000_000).to_numpy()   # time64[us] -> hour
hourly = t.drop(["time"]).to_pandas(date_as_object=False)
hourly["timestamp_utc"] = hourly.pop("date").dt.tz_localize("UTC") + pd.to_timedelta(hour_utc, unit="h")
del t, hour_utc

sensors = pd.read_parquet(FINAL / "sensor_steepness.parquet")
gaps = pd.read_parquet(INTERIM / "missing_hours.parquet")

print(f"{len(hourly):,} sensor-hours, {hourly.location_id.nunique()} sensors, "
      f"{hourly.timestamp_utc.min():%Y-%m-%d} to {hourly.timestamp_utc.max():%Y-%m-%d} (UTC)")

## 3. Slice columns

| Slice | Groups | Why it could hide bias |
|---|---|---|
| `precinct` | UniMelb edge, Carlton, North Melbourne, CBD and other | the transfer-test precincts are small; a model can look good overall and be poor there |
| `sensor_volume` | quiet / medium / busy (thirds of sensors by their median count) | sensors were placed where traffic matters, so quiet paths are thin in the data |
| `sensor_history` | first year with counts: 2009-2013 / 2014-2018 / 2019 on | early years have few sensors, so they're dominated by a handful of streets |
| `time_band` | night 0-5, morning 6-9, midday 10-15, evening 16-19, late 20-23 (local) | most people walk in the day; night errors get hidden |
| `day_type` | weekday / weekend (local) | different patterns |
| `season` | summer (Dec-Feb), autumn, winter, spring | for checking only: month is never a model feature |
| `era` | pre-COVID, COVID, recovery, post Metro Tunnel | crowds changed sharply at each break |
| `weather_rain` | dry / wet / no weather | rainy hours are a minority |
| `temp_band` | below 10, 10-20, 20-30, 30 and above (°C), no weather | hot and cold extremes are rare |
| `source` | archive / wayback / live | different collection; Wayback is a third-party copy |
| `slope_band` | below 1%, 1-3%, 3-5%, 5% and above | steep streets are rare (one sensor above 20%) |
| `place_source` | footpath / LiDAR | 2 sensors use a different method for slope and elevation |
| `outage_flag` | flagged / not flagged | suspected broken-sensor zeros |

In [ ]:
local = hourly.timestamp_utc.dt.tz_convert(TZ)
local_date = local.dt.tz_localize(None).dt.normalize()

hourly["time_band"] = pd.cut(local.dt.hour, [-1, 5, 9, 15, 19, 23],
                             labels=["night 0-5", "morning 6-9", "midday 10-15", "evening 16-19", "late 20-23"])
hourly["day_type"] = np.where(local.dt.day_of_week >= 5, "weekend", "weekday")
hourly["season"] = local.dt.month.map({12: "summer", 1: "summer", 2: "summer", 3: "autumn", 4: "autumn", 5: "autumn",
                                       6: "winter", 7: "winter", 8: "winter", 9: "spring", 10: "spring", 11: "spring"})
hourly["era"] = pd.cut(local_date,
                       [pd.Timestamp.min, pd.Timestamp(COVID_START) - pd.Timedelta(days=1), pd.Timestamp(COVID_END),
                        pd.Timestamp(METRO_TUNNEL) - pd.Timedelta(days=1), pd.Timestamp.max],
                       labels=["pre-COVID", "COVID", "recovery", "post Metro Tunnel"])
hourly["weather_rain"] = np.select([hourly.rain_mm.isna(), hourly.rain_mm > RAIN_MM], ["no weather", "wet"], "dry")
hourly["temp_band"] = pd.cut(hourly.temp_c, [-np.inf, 10, 20, 30, np.inf], right=False,
                             labels=["<10°C", "10-20°C", "20-30°C", "≥30°C"]).cat.add_categories("no weather")
hourly["temp_band"] = hourly.temp_band.fillna("no weather")
hourly["outage_flag"] = np.where(hourly.suspect_outage_zero, "flagged", "not flagged")
hourly["year"] = local.dt.year

# Per-sensor slices, worked out once per sensor and then copied onto its rows.
per_sensor = hourly[~hourly.suspect_outage_zero].groupby("location_id").agg(
    median_count=("pedestriancount", "median"), first_year=("year", "min"))
per_sensor["sensor_volume"] = pd.qcut(per_sensor.median_count, 3, labels=["quiet", "medium", "busy"])
per_sensor["sensor_history"] = pd.cut(per_sensor.first_year, [0, 2013, 2018, 9999],
                                      labels=["since 2009-2013", "since 2014-2018", "since 2019+"])
precinct = {sid: name for name, ids in PRECINCTS.items() for sid in ids}
per_sensor["precinct"] = per_sensor.index.map(lambda sid: precinct.get(sid, "CBD and other"))
place = sensors.set_index("location_id")
per_sensor["slope_band"] = pd.cut(place.slope_pct.reindex(per_sensor.index), [-np.inf, 1, 3, 5, np.inf], right=False,
                                  labels=["<1%", "1-3%", "3-5%", "≥5%"])
per_sensor["place_source"] = place.place_source.reindex(per_sensor.index).map(
    {"footpath_steepness": "footpath", "ga_dem_5m": "LiDAR"})
hourly = hourly.join(per_sensor[["sensor_volume", "sensor_history", "precinct", "slope_band", "place_source"]],
                     on="location_id")

print("sensors per precinct:", per_sensor.precinct.value_counts().to_dict())
print("sensor median count cut-offs (quiet | medium | busy):",
      per_sensor.groupby("sensor_volume", observed=True).median_count.agg(["min", "max"]).round(0).to_dict("index"))

## 4. Slice report

For every slice: how many rows and sensors it has, its share of all rows, how its counts are distributed, and how much of it is zeros, flagged outages or missing weather. `issues` names any problem found, using the thresholds in Setup.

Outage-flagged rows are excluded from the count statistics (median, mean, 90th percentile, zeros), since they're suspected sensor faults, not real counts. They're still counted in `rows` and `outage_share`.

In [ ]:
SLICES = ["precinct", "sensor_volume", "sensor_history", "time_band", "day_type", "season", "era",
          "weather_rain", "temp_band", "source", "slope_band", "place_source", "outage_flag"]

hourly["is_zero"] = hourly.pedestriancount.eq(0)
hourly["no_weather"] = hourly.temp_c.isna()
valid = ~hourly.suspect_outage_zero
overall_median = hourly.pedestriancount[valid].median()


def slice_table(col: str) -> pd.DataFrame:
    g_all = hourly.groupby(col, observed=True)
    g = hourly[valid].groupby(col, observed=True)
    t = pd.DataFrame({
        "rows": g_all.size(),
        "share_of_rows": g_all.size() / len(hourly),
        "sensors": g_all.location_id.nunique(),
        "median_count": g.pedestriancount.median(),
        "mean_count": g.pedestriancount.mean(),
        "p90_count": g.pedestriancount.quantile(0.9),
        "zero_share": g.is_zero.mean(),
        "outage_share": g_all.suspect_outage_zero.mean(),
        "no_weather_share": g_all.no_weather.mean(),
    })
    ratio = t.median_count / overall_median
    flags = pd.Series("", index=t.index)
    flags[t.share_of_rows < SMALL_SHARE] += "small; "
    flags[(ratio > MEDIAN_RATIO) | (ratio < 1 / MEDIAN_RATIO)] += "median far from overall; "
    flags[t.no_weather_share > MISSING_WEATHER] += "missing weather; "
    t["issues"] = flags.str.rstrip("; ")
    return t.rename_axis("group").reset_index().assign(slice=col)


report = pd.concat([slice_table(c) for c in SLICES], ignore_index=True)
report = report[["slice", "group", *[c for c in report.columns if c not in ("slice", "group")]]]
report.to_csv(REPORTS / "slice_report.csv", index=False)
print(f"overall median count: {overall_median:.0f} people/hour; report saved to {REPORTS / 'slice_report.csv'}")

fmt = {"share_of_rows": "{:.1%}", "zero_share": "{:.1%}", "outage_share": "{:.2%}", "no_weather_share": "{:.2%}",
       "median_count": "{:.0f}", "mean_count": "{:.0f}", "p90_count": "{:.0f}", "rows": "{:,}"}
for col in SLICES:
    print(f"\n=== {col}")
    print(report[report.slice == col].drop(columns="slice").to_string(index=False, formatters={k: v.format for k, v in fmt.items()}))

## 5. Coverage over time and per sensor

Two kinds of under-representation that a single slice table can hide:
- **over time:** how many sensors were counting each year (early years rest on a few streets);
- **per sensor:** how complete each sensor's record is between its first and last hour (`missing_hours` from combine).

In [ ]:
by_year = hourly.groupby("year").agg(rows=("location_id", "size"), sensors=("location_id", "nunique"))
by_year["share_of_rows"] = (by_year.rows / len(hourly)).round(3)
print(by_year.to_string())
by_year.to_csv(REPORTS / "coverage_by_year.csv")

In [ ]:
missing = gaps.groupby("location_id").n_hours.sum()
completeness = pd.DataFrame({"rows": hourly.groupby("location_id").size()})
completeness["missing_hours"] = missing.reindex(completeness.index).fillna(0).astype(int)
completeness["completeness"] = completeness.rows / (completeness.rows + completeness.missing_hours)
completeness = completeness.join(per_sensor[["precinct", "sensor_volume"]])
completeness.sort_values("completeness").to_csv(REPORTS / "sensor_completeness.csv")

print(completeness.completeness.describe(percentiles=[0.1, 0.25, 0.5]).round(3).to_string())
print("\nleast complete sensors:")
print(completeness.sort_values("completeness").head(10).round(3).to_string())
print("\nmedian completeness by precinct:", completeness.groupby("precinct").completeness.median().round(3).to_dict())

### Combined slices

Problems often show up only when two slices meet, e.g. a quiet sensor at night in the rain. Two examples: the share of rows, and the median count, for every combination.

In [ ]:
for a, b in [("time_band", "weather_rain"), ("precinct", "era")]:
    share = pd.crosstab(hourly[a], hourly[b], normalize=True)
    med = hourly[valid].pivot_table(index=a, columns=b, values="pedestriancount", aggfunc="median", observed=True)
    print(f"\n=== {a} x {b}: share of all rows")
    print((share * 100).round(2).astype(str).add("%").to_string())
    print(f"=== {a} x {b}: median count")
    print(med.round(0).to_string())

## 6. Findings

Every slice that tripped a threshold. Use this list for the write-up below.

In [ ]:
flagged = report[report["issues"] != ""][["slice", "group", "rows", "share_of_rows", "sensors", "median_count", "issues"]]
print(flagged.to_string(index=False, formatters={"share_of_rows": "{:.1%}".format, "rows": "{:,}".format,
                                                  "median_count": "{:.0f}".format}))
summary = {
    "rows": len(hourly), "sensors": int(hourly.location_id.nunique()), "overall_median_count": float(overall_median),
    "thresholds": {"small_share": SMALL_SHARE, "median_ratio": MEDIAN_RATIO, "missing_weather": MISSING_WEATHER},
    "flagged_slices": flagged.assign(share_of_rows=flagged.share_of_rows.round(4)).to_dict("records"),
}
(REPORTS / "stage1_summary.json").write_text(json.dumps(summary, indent=2, default=str))

### Mitigation options (to decide once a model's error can be checked per slice)

Under-representation in the data only matters if a model is worse on that slice, so these findings are candidates: confirm or rule them out by comparing a trained model's error on held-out sensors across the same slices.

| If the model is worse on... | Options |
|---|---|
| small slices (a precinct, LiDAR sensors, steep sensors) | report them separately rather than inside the average; weight their rows up in training |
| quiet sensors | weight sensors equally (each sensor's rows sum to the same weight) so busy sensors don't dominate; check relative error, not only absolute |
| early years / sensors with short histories | weight by sensor-year; or evaluate on a recent window only and say so |
| night, rain, extreme temperature | re-sample or weight those hours up; check whether the weather features are informative there |
| COVID / post Metro Tunnel | flag or hold out the era; the team already treats both as structural breaks |
| outage-flagged rows | drop them from training and evaluation (the counts are suspected faults) |

Whatever is changed, record the trade-off: each slice's error before and after, and the change in overall error.

### Write-up (fill in after running)

- **Slices checked:** the 13 in section 3.
- **Under-represented slices found:** ...
- **Slices whose counts differ most from overall:** ...
- **Data-quality gaps (missing weather, outages, incomplete sensors):** ...
- **Planned mitigation and why:** ...